# 🫁 Module 3: Chest X-Ray Pneumonia Detection
**Dataset:** Paul Mooney Chest X-Ray (Pneumonia) via Kaggle

**Model:** EfficientNetB3 Transfer Learning + Grad-CAM

**Output:** `xray_model.h5` → place in `ai-server/models/`

---
⚠️ **Recommended:** Run on Kaggle Notebooks or Google Colab (free T4 GPU)

In [ ]:
!pip install -q kaggle tensorflow matplotlib numpy opencv-python Pillow

In [ ]:
# ─── GPU CHECK ────────────────────────────────────────────────────────────────
import tensorflow as tf
print('TF Version:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))

# Enable mixed precision for faster training on GPU
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
print('Mixed precision enabled')

In [ ]:
# ─── STEP 2: Download Dataset ─────────────────────────────────────────────────
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p ./data --unzip
!ls ./data/chest_xray/

In [ ]:
# ─── STEP 3: Explore Dataset Structure ────────────────────────────────────────
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

BASE_DIR = './data/chest_xray'

for split in ['train', 'val', 'test']:
    for cls in ['NORMAL', 'PNEUMONIA']:
        path = os.path.join(BASE_DIR, split, cls)
        count = len(os.listdir(path))
        print(f'{split}/{cls}: {count} images')

# Visualize samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, cls in enumerate(['NORMAL', 'PNEUMONIA']):
    imgs = os.listdir(f'{BASE_DIR}/train/{cls}')[:4]
    for j, img_name in enumerate(imgs):
        img = mpimg.imread(f'{BASE_DIR}/train/{cls}/{img_name}')
        axes[i][j].imshow(img, cmap='gray')
        axes[i][j].set_title(cls, fontsize=10)
        axes[i][j].axis('off')
plt.suptitle('Sample X-Ray Images', fontsize=14)
plt.tight_layout()
plt.savefig('./data/xray_samples.png', dpi=100)
plt.show()

In [ ]:
# ─── STEP 4: Data Generators with Augmentation ────────────────────────────────
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Strong augmentation for training
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.9, 1.1],
    fill_mode='nearest'
)

val_gen = ImageDataGenerator(rescale=1./255)

train_ds = train_gen.flow_from_directory(
    f'{BASE_DIR}/train', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', color_mode='rgb'
)

val_ds = val_gen.flow_from_directory(
    f'{BASE_DIR}/val', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', color_mode='rgb'
)

test_ds = val_gen.flow_from_directory(
    f'{BASE_DIR}/test', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', color_mode='rgb', shuffle=False
)

print('Classes:', train_ds.class_indices)

In [ ]:
# ─── STEP 5: Build EfficientNetB3 Transfer Learning Model ────────────────────
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras import layers, models, regularizers

def build_model():
    base = EfficientNetB3(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3)
    )
    base.trainable = False  # Freeze initially
    
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu', 
                     kernel_regularizer=regularizers.l2(0.01))(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid', dtype='float32')(x)  # float32 for mixed precision
    
    return tf.keras.Model(inputs, outputs), base

model, base_model = build_model()
model.summary()

In [ ]:
# ─── STEP 6: Phase 1 - Train Head Only ────────────────────────────────────────
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np

# Class weights for imbalanced dataset
total = train_ds.samples
n_normal = train_ds.classes.tolist().count(0)
n_pneumonia = train_ds.classes.tolist().count(1)
class_weight = {0: total/(2*n_normal), 1: total/(2*n_pneumonia)}
print('Class weights:', class_weight)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_auc', mode='max'),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-7, monitor='val_loss'),
    ModelCheckpoint('./output/xray_best_phase1.h5', save_best_only=True, monitor='val_auc', mode='max')
]

os.makedirs('./output', exist_ok=True)

history1 = model.fit(
    train_ds, epochs=15,
    validation_data=val_ds,
    callbacks=callbacks,
    class_weight=class_weight
)

In [ ]:
# ─── STEP 7: Phase 2 - Fine-Tune (Unfreeze top layers) ───────────────────────
# Unfreeze top 30 layers of EfficientNetB3
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # Lower LR for fine-tuning
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

callbacks2 = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_auc', mode='max'),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-8, monitor='val_loss'),
    ModelCheckpoint('./output/xray_best_final.h5', save_best_only=True, monitor='val_auc', mode='max')
]

history2 = model.fit(
    train_ds, epochs=20,
    validation_data=val_ds,
    callbacks=callbacks2,
    class_weight=class_weight
)

In [ ]:
# ─── STEP 8: Evaluate on Test Set ─────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred_prob = model.predict(test_ds).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)
y_true = test_ds.classes

print('=== TEST RESULTS ===')
print(f'AUC-ROC: {roc_auc_score(y_true, y_pred_prob):.4f}')
print(classification_report(y_true, y_pred, target_names=['NORMAL', 'PNEUMONIA']))

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

all_acc = history1.history['accuracy'] + history2.history['accuracy']
all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']

axes[0].plot(all_acc, label='Train')
axes[0].plot(all_val_acc, label='Val')
axes[0].set_title('Accuracy')
axes[0].legend()

cm = confusion_matrix(y_true, y_pred)
import seaborn as sns
sns.heatmap(cm, annot=True, fmt='d', ax=axes[1],
            xticklabels=['NORMAL','PNEUMONIA'], yticklabels=['NORMAL','PNEUMONIA'])
axes[1].set_title('Confusion Matrix')

plt.tight_layout()
plt.savefig('./data/xray_results.png', dpi=100)
plt.show()

In [ ]:
# ─── STEP 9: Save Final Model ─────────────────────────────────────────────────
model.save('./output/xray_model.h5')
print('✅ Saved: ./output/xray_model.h5')
print('\n📁 COPY THIS FILE TO: healthvision-ai/ai-server/models/xray_model.h5')

In [ ]:
# ─── STEP 10: Grad-CAM Visualization (matches gradcam.py in ai-server) ────────
import cv2
import numpy as np
from tensorflow.keras.models import load_model

def get_gradcam(model, img_array, last_conv_layer='top_conv'):
    grad_model = tf.keras.Model(
        [model.inputs],
        [model.get_layer('efficientnetb3').get_layer(last_conv_layer).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap).numpy()
    heatmap = np.maximum(heatmap, 0) / (heatmap.max() + 1e-8)
    return heatmap

# Test on a sample image
sample_imgs = os.listdir(f'{BASE_DIR}/test/PNEUMONIA')[:1]
img_path = f'{BASE_DIR}/test/PNEUMONIA/{sample_imgs[0]}'
img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
img_array = tf.keras.utils.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, 0)

pred = model.predict(img_array)[0][0]
print(f'Prediction: {pred:.4f} → {"PNEUMONIA" if pred > 0.5 else "NORMAL"}')